# TP 6 : IA & Santé - NLP 📑
(CORRECTION DÉTAILLÉE)

**Objectif :** Classifer des rapports médicaux avec TF-IDF.

In [ ]:
# === CORRECTION COMPLÈTE : CLASSIFICATION DE TEXTES MÉDICAUX ===

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from wordcloud import WordCloud

print("="*60)
print("TP6 : Classification de textes avec TF-IDF")
print("="*60)

# ── ÉTAPE 1 : Charger le dataset ──────────────────────────────
print("\n1️⃣  CHARGEMENT DES DONNÉES\n")

# On prend sci.med (textes médicaux) vs sci.space (textes spatiaux)
# pour créer une tâche de classification binaire claire
categories = ['sci.med', 'sci.space']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                remove=('headers', 'footers', 'quotes'))

print(f"✓ Train : {len(train_data.data)} textes")
print(f"✓ Test  : {len(test_data.data)} textes")
print(f"  Catégories : {train_data.target_names}\n")

print("Exemple de texte médical :")
print(train_data.data[0][:300], "...\n")

# ── ÉTAPE 2 : Vectorisation TF-IDF ───────────────────────────
print("2️⃣  VECTORISATION TF-IDF\n")

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)

X_train = vectorizer.fit_transform(train_data.data)
X_test  = vectorizer.transform(test_data.data)
y_train = train_data.target
y_test  = test_data.target

print(f"✓ Matrice d'entraînement : {X_train.shape}")
print(f"  (documents × features TF-IDF)\n")

# ── ÉTAPE 3 : WordCloud des mots importants ───────────────────
print("3️⃣  VISUALISATION : WORDCLOUD\n")

# On récupère le score TF-IDF moyen de chaque mot sur les textes médicaux
med_indices = np.where(y_train == 0)[0]   # 0 = sci.med
X_med = X_train[med_indices]
mean_tfidf = np.asarray(X_med.mean(axis=0)).flatten()
feature_names = vectorizer.get_feature_names_out()

# Dictionnaire mot → poids
word_weights = dict(zip(feature_names, mean_tfidf))

wordcloud = WordCloud(width=800, height=400, background_color='white',
                      colormap='Blues').generate_from_frequencies(word_weights)

plt.figure(figsize=(12, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Mots les plus importants dans sci.med (TF-IDF)", fontsize=14)
plt.tight_layout()
plt.show()

# ── ÉTAPE 4 : Entraîner un classifieur ───────────────────────
print("4️⃣  ENTRAÎNEMENT DU CLASSIFIEUR\n")

# Naive Bayes (rapide, très efficace en NLP)
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_score = nb.score(X_test, y_test)
print(f"Naive Bayes  → précision : {nb_score:.3f}")

# SVM (en option, généralement plus puissant)
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)
svm_score = svm.score(X_test, y_test)
print(f"SVM linéaire → précision : {svm_score:.3f}\n")

# Rapport détaillé avec le meilleur modèle
best_model = nb if nb_score >= svm_score else svm
best_name  = "Naive Bayes" if nb_score >= svm_score else "SVM"
print(f"✓ Meilleur modèle : {best_name}\n")

y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=test_data.target_names))

# ── ÉTAPE 5 : Tester sur une phrase inventée ─────────────────
print("5️⃣  TEST SUR PHRASES INVENTÉES\n")

phrases = [
    "The patient was diagnosed with pneumonia and prescribed antibiotics.",
    "The rocket successfully launched from Cape Canaveral into orbit.",
    "Blood pressure was elevated, surgery recommended next week.",
    "Astronauts aboard the space station conducted experiments in microgravity.",
]

X_phrases = vectorizer.transform(phrases)
predictions = best_model.predict(X_phrases)

for phrase, pred in zip(phrases, predictions):
    label = test_data.target_names[pred]
    icon  = "🩺" if pred == 0 else "🚀"
    print(f"  {icon} [{label}]")
    print(f"     \"{phrase}\"\n")